# Retrieval-Augmented Generation (RAG) from Scratch with Gemini

This notebook goes with the article [RAG from Scratch](article.md). We build a complete question-answering assistant **without any framework**, so every step stays visible:

```
load -> split -> embed -> store -> retrieve -> prompt -> generate
```

**The knowledge base.** The `data/` folder holds five short internal documents from **Brightwave Labs**, a *fictional* company invented for this tutorial: an AI usage policy, an architecture document, an FAQ, and so on. Because the company isn't real, no language model has ever seen these documents. Any correct answer must come from **retrieval**, not from the model's memory. That makes the effect of RAG easy to see.

**Setup:** get a free key at <https://aistudio.google.com/apikey>, put it in a `.env` file in the repository root (see `.env.example`), and run `pip install -r requirements.txt`. The helper code lives in the `utils/` package next to this notebook.

In [1]:
from pathlib import Path

from utils.chatmodel import ChatGemini
from utils.embedding import EmbeddingModel, get_client
from utils.prompts import SystemRolePrompt, UserRolePrompt
from utils.text_utils import CharacterTextSplitter, TextFileLoader
from utils.vectordatabase import VectorDatabase

DATA_DIR = Path("data") if Path("data").exists() else Path(__file__).resolve().parent / "data"
client = get_client()  # one client, shared by the embedding model and the chat model

## Step 1: Load the documents

`TextFileLoader` reads every `.txt` file in a folder. Each `Document` remembers its **source file**, so the assistant can cite where an answer came from.

In [2]:
documents = TextFileLoader(DATA_DIR).load_documents()

print(f"Loaded {len(documents)} documents:")
for doc in documents:
    print(f"  {doc.source:32} {len(doc.text):>5} characters")

Loaded 5 documents:
  01-company-overview.txt           2286 characters
  02-ai-usage-policy.txt            2779 characters
  03-atlas-architecture.txt         2728 characters
  04-evaluation-and-quality.txt     2657 characters
  05-employee-faq.txt               2153 characters


## Step 2: Split them into chunks

Whole documents are too coarse to retrieve: a question about vacation days doesn't need the entire FAQ. So we cut each document into **chunks** of about 1,000 characters.

Neighbouring chunks **overlap** by 200 characters. Without the overlap, a sentence on a boundary would be cut in half and might not make sense in either chunk. The output below confirms that the last 200 characters of chunk 0 open chunk 1.

Notice that the cuts land mid-word. A fixed character count is the simplest possible splitter; production systems split on paragraphs or sentences instead.

In [3]:
splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(documents)
print(f"{len(documents)} documents -> {len(chunks)} chunks\n")

first, second = chunks[0], chunks[1]
shared = first.text[-splitter.chunk_overlap:]
print("Last 200 characters of chunk 0 == first 200 of chunk 1?", second.text.startswith(shared))
print("Shared text:", repr(shared))

5 documents -> 18 chunks

Last 200 characters of chunk 0 == first 200 of chunk 1? True
Shared text: "de CareRoute that turns a nurse's voice memo into a structured visit summary. A nurse must review and approve every summary before it is saved.\n3. Support Drafts, which suggests replies to customer ti"


## Step 3: Embed the chunks and store them

Every chunk becomes an embedding vector (see [01-Embeddings](../01-Embeddings/)). All the chunks go to the API in **one batched request**, which matters on the free tier. Our `VectorDatabase` simply keeps the vectors in a NumPy matrix, one row per chunk.

In [4]:
embedding_model = EmbeddingModel(client=client)
vector_db = VectorDatabase(embedding_model).build_from_documents(chunks)
print("Vector matrix shape:", vector_db.vectors.shape, "(chunks x dimensions)")

Vector matrix shape: (18, 3072) (chunks x dimensions)


## Step 4: Retrieve

To answer a question, we embed it (with `task_type="RETRIEVAL_QUERY"`) and return the chunks with the highest cosine similarity. This step alone is **semantic search**.

The preview shows how each chunk *starts*. A chunk can open with one topic and contain the answer further down, which is why we also check whether it actually defines the rule.

In [5]:
question = "What is the Two-Source Rule?"

for chunk, score in vector_db.search_by_text(question, k=3):
    contains = "yes" if "Two-Source Rule" in chunk.text else "no "
    preview = " ".join(chunk.text.split())[:70]
    print(f"{score:.3f}  [{chunk.source}]  defines the rule: {contains}  | starts: {preview}...")

0.699  [02-ai-usage-policy.txt]  defines the rule: yes  | starts: Amber data: internal information, such as code, roadmaps and internal ...
0.626  [04-evaluation-and-quality.txt]  defines the rule: no   | starts: ays "paid leave". Fix: hybrid search, which combines keyword search wi...
0.625  [02-ai-usage-policy.txt]  defines the rule: no   | starts: It exists because language models can produce fluent, confident text t...


## Step 5: What happens *without* retrieval?

Before adding the generation step, let's ask the language model the same question directly, with no context. It has never seen Brightwave's documents, so it can only guess or decline.

In [6]:
chat_model = ChatGemini(client=client)
print("Chat model:", chat_model.model_name, "\n")

no_rag_answer = chat_model.run([
    {"role": "user", "content": "At Brightwave Labs, what is the Two-Source Rule?"}
])
print(no_rag_answer)

Chat model: gemini-3.5-flash-lite 



I do not have access to internal company policies, proprietary documents, or specific operational procedures for Brightwave Labs, so I cannot provide the definition or context of the "Two-Source Rule" used within that organization.


## Step 6: The RAG prompt

The prompt is where retrieval and generation meet. The **system prompt** sets the rules: answer only from the context, cite sources, and say "I don't know" otherwise. The **user prompt** carries the retrieved chunks and the question.

In [7]:
RAG_SYSTEM_TEMPLATE = """You are Atlas, an internal assistant for Brightwave Labs.
Answer the question using ONLY the context provided.
After each fact, cite its source file in square brackets, for example [02-ai-usage-policy.txt].
If the context does not contain the answer, reply exactly: "I don't know based on the available documents."
Keep answers short and factual."""

RAG_USER_TEMPLATE = """Context:
{context}

Question:
{question}"""

system_prompt = SystemRolePrompt(RAG_SYSTEM_TEMPLATE)
user_prompt = UserRolePrompt(RAG_USER_TEMPLATE)
print("User prompt variables:", user_prompt.get_input_variables())

User prompt variables: ['context', 'question']


## Step 7: Put it all together

The `RAGPipeline` class runs the full loop for each question: **retrieve** the top-*k* chunks, **format** them into the prompt with their source names, and **generate** an answer. It also returns the chunks it used, so we can check the answer against them.

In [8]:
class RAGPipeline:
    def __init__(self, llm: ChatGemini, vector_db: VectorDatabase, k: int = 4):
        self.llm = llm
        self.vector_db = vector_db
        self.k = k

    def run(self, question: str) -> dict:
        retrieved = self.vector_db.search_by_text(question, k=self.k)
        context = "\n\n".join(f"[{chunk.source}]\n{chunk.text}" for chunk, _ in retrieved)
        messages = [
            system_prompt.create_message(),
            user_prompt.create_message(context=context, question=question),
        ]
        return {"answer": self.llm.run(messages), "context": retrieved}


rag = RAGPipeline(chat_model, vector_db)
result = rag.run("What is the Two-Source Rule?")
print(result["answer"])

The Two-Source Rule states that any fact produced by an AI system that will be shown to a customer or used in a clinical context must be confirmed by two independent sources before it is used [02-ai-usage-policy.txt]. One of the sources may be the document the AI cited, but the second must be a human expert or a system of record (for example, the patient's care plan) [02-ai-usage-policy.txt].


Compare this with Step 5: the answer is now **specific, correct and cited**, because the model was handed the right passage.

## Step 8: More questions, including one it should refuse

The last question is general knowledge that is **not** in the documents. A well-grounded assistant should decline instead of answering from memory.

In [9]:
questions = [
    "What chunk size does Atlas use, and why was it chosen?",
    "Who has to approve a new AI model vendor, and in what order?",
    "How many vacation days do I get after four years?",
    "Who won the 2022 FIFA World Cup?",  # not in the documents
]

for q in questions:
    result = rag.run(q)
    sources = sorted({chunk.source for chunk, _ in result["context"]})
    print(f"Q: {q}\nA: {result['answer'].strip()}\n   retrieved from: {', '.join(sources)}\n")

Q: What chunk size does Atlas use, and why was it chosen?
A: Atlas uses a chunk size of about 800 tokens (with an overlap of 120 tokens) [03-atlas-architecture.txt]. This size was chosen because it gave the best answer quality on the evaluation set, as smaller chunks lost context and larger chunks mixed several topics together [03-atlas-architecture.txt].
   retrieved from: 03-atlas-architecture.txt, 04-evaluation-and-quality.txt



Q: Who has to approve a new AI model vendor, and in what order?
A: Adding a new model provider requires three approvals, in this order:
1. A security review by the Security team (which checks data retention and encryption) [02-ai-usage-policy.txt].
2. A privacy review by the Data Protection Officer [02-ai-usage-policy.txt].
3. Final approval by the Responsible AI Council [02-ai-usage-policy.txt].
   retrieved from: 01-company-overview.txt, 02-ai-usage-policy.txt



Q: How many vacation days do I get after four years?
A: After three years at Brightwave you receive two extra days [05-employee-faq.txt]. Full-time employees receive 25 days of paid leave per year [05-employee-faq.txt]. Therefore, after four years you receive 27 days of paid leave per year (25 days plus two extra days).
   retrieved from: 04-evaluation-and-quality.txt, 05-employee-faq.txt



Q: Who won the 2022 FIFA World Cup?
A: I don't know based on the available documents.
   retrieved from: 01-company-overview.txt, 03-atlas-architecture.txt, 04-evaluation-and-quality.txt, 05-employee-faq.txt



## Recap and next steps

You built a working RAG system in about 100 lines of plain Python:

| Step | Code | What it does |
| --- | --- | --- |
| Load | `TextFileLoader` | reads files and keeps their names for citations |
| Split | `CharacterTextSplitter` | makes overlapping chunks small enough to retrieve precisely |
| Embed + store | `EmbeddingModel`, `VectorDatabase` | turns chunks into vectors in one batched request |
| Retrieve | `search_by_text` | finds the chunks closest in meaning to the question |
| Prompt | `SystemRolePrompt`, `UserRolePrompt` | sets the grounding rules and injects the context |
| Generate | `ChatGemini` | writes an answer that cites its sources |

Ways to improve it, several of which the Brightwave documents themselves describe:

- **Smarter chunking**: split on paragraphs or sentences instead of a fixed character count.
- **Hybrid search**: add keyword search (BM25) for acronyms and exact names.
- **Reranking**: retrieve 20 chunks, then let a stronger model keep the best 5.
- **Evaluation**: track retrieval hit rate and answer faithfulness on a fixed set of test questions before changing anything.